<a href="https://colab.research.google.com/github/TahaNawazkhan/Artificial-intelligence-/blob/main/lab%203%20task%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import matplotlib
matplotlib.use("Agg")                       # save to files, no window needed
import matplotlib.pyplot as plt
import networkx as nx

OUT_DIR = "iddfs_frames"


# ---------------- Tree ----------------
class Node:
    def __init__(self, name):
        self.name = name
        self.children = []

    def add(self, *kids):
        self.children.extend(kids)
        return self


def build_sample_tree():
    #               A
    #        /      |      \
    #       B       C       D
    #     / | \    / \      |
    #    E  F  G  H   I     J
    #   /\     |      |    /|\
    #  K  L    M      N   O P Q
    n = {c: Node(c) for c in "ABCDEFGHIJKLMNOPQ"}
    n["A"].add(n["B"], n["C"], n["D"])
    n["B"].add(n["E"], n["F"], n["G"])
    n["C"].add(n["H"], n["I"])
    n["D"].add(n["J"])
    n["E"].add(n["K"], n["L"])
    n["G"].add(n["M"])
    n["I"].add(n["N"])
    n["J"].add(n["O"], n["P"], n["Q"])
    return n["A"]


# ---------------- Node tree -> networkx graph ----------------
def tree_to_graph(root):
    G = nx.DiGraph()
    stack = [root]
    while stack:
        node = stack.pop()
        G.add_node(node.name)
        for child in node.children:
            G.add_edge(node.name, child.name)
            stack.append(child)
    return G


# ---------------- Hierarchical layout (no graphviz needed) ----------------
def tree_layout(root):
    pos, next_x = {}, [0]

    def place(node, depth):
        if not node.children:
            x = next_x[0]
            next_x[0] += 1
        else:
            xs = [place(c, depth + 1) for c in node.children]
            x = sum(xs) / len(xs)           # parent centred over children
        pos[node.name] = (x, -depth)
        return x
    place(root, 0)
    return pos


# ---------------- Depth-limited search that records visited nodes ----------------
def dls_record(node, goal, limit, visited):
    visited.append(node.name)
    if node.name == goal:
        return True
    if limit == 0:
        return False
    for child in node.children:
        if dls_record(child, goal, limit - 1, visited):
            return True
    return False


# ---------------- Drawing ----------------
def draw_frame(G, pos, visited, limit, goal, found, total_so_far, path):
    colors = ["green" if n in visited else "lightgrey" for n in G.nodes()]
    plt.figure(figsize=(10, 6))
    nx.draw(G, pos, with_labels=True, node_color=colors, node_size=900,
            font_size=11, font_weight="bold", arrows=False,
            edgecolors="black")
    plt.title(f"IDDFS  |  depth limit = {limit}  |  goal = {goal}\n"
              f"visited this iteration: {len(visited)}   "
              f"cumulative total: {total_so_far}"
              f"{'   -> GOAL FOUND' if found else ''}")
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()


def visualise_iddfs(root, goal, max_depth=50):
    os.makedirs(OUT_DIR, exist_ok=True)
    G = tree_to_graph(root)
    pos = tree_layout(root)
    total = 0
    for limit in range(max_depth + 1):
        visited = []
        found = dls_record(root, goal, limit, visited)
        total += len(visited)
        path = os.path.join(OUT_DIR, f"iddfs_goal_{goal}_limit_{limit}.png")
        draw_frame(G, pos, set(visited), limit, goal, found, total, path)
        print(f"limit {limit}: visited {len(visited):>2} nodes "
              f"(cumulative {total}) -> {path}")
        if found or len(visited) == len(G):  # goal found, or whole tree covered
            break


if __name__ == "__main__":
    visualise_iddfs(build_sample_tree(), goal="Q")

limit 0: visited  1 nodes (cumulative 1) -> iddfs_frames/iddfs_goal_Q_limit_0.png
limit 1: visited  4 nodes (cumulative 5) -> iddfs_frames/iddfs_goal_Q_limit_1.png
limit 2: visited 10 nodes (cumulative 15) -> iddfs_frames/iddfs_goal_Q_limit_2.png
limit 3: visited 17 nodes (cumulative 32) -> iddfs_frames/iddfs_goal_Q_limit_3.png
